# GraphRAG Intro

# 01. GraphRAG 개념이해

## 1. GraphRAG란?

**GraphRAG(Graph Retrieval-Augmented Generation)** 는 문서에서 중요한 **개체(Entity)** 와 개체 사이의 **관계(Relationship)** 를 추출하여 지식 그래프를 만들고, 질문과 관련된 그래프 정보를 검색해 LLM의 답변 생성에 활용하는 방식입니다.

- **노드(Node)**: 사람, 장소, 조직, 아이템, 사건과 같은 개체
- **엣지(Edge)**: 개체 사이의 관계
- **속성(Property)**: 개체나 관계에 포함된 세부 정보

예를 들어 `용사가 전설의 검을 사용한다`라는 문장은 다음과 같은 그래프로 표현할 수 있습니다.

```text
(용사) --[사용한다]--> (전설의 검)
```

## 2. 일반 RAG와 GraphRAG의 차이

| 구분 | 일반 RAG | GraphRAG |
|---|---|---|
| 데이터 표현 | 문서를 일정 크기의 청크와 벡터로 저장 | 개체와 관계를 그래프로 저장 |
| 검색 기준 | 질문과 문장의 의미적 유사도 | 연결된 개체, 관계, 경로, 하위 그래프 |
| 강점 | 관련 문장이나 단일 사실 검색 | 여러 문서에 흩어진 관계와 연결 구조 탐색 |
| 대표 질문 | “마법사의 능력은 무엇인가?” | “마법사와 용의 봉인에는 어떤 관계가 있는가?” |

일반 RAG는 질문과 의미가 비슷한 문장을 빠르게 찾는 데 강하지만, 여러 청크에 나뉜 정보를 연결하거나 개체 사이의 다단계 관계를 추론하는 데 한계가 있습니다. GraphRAG는 그래프의 연결 구조를 사용하여 이러한 관계 중심 질문에 필요한 문맥을 구성합니다.

## 3. GraphRAG의 처리 흐름

```text
원본 문서
   ↓
개체·관계 추출
   ↓
지식 그래프 구축
   ↓
질문에서 핵심 개체 식별
   ↓
관련 노드·관계·경로 검색
   ↓
검색 결과를 문맥으로 구성
   ↓
LLM 답변 생성
```

예를 들어 `용을 쓰러뜨리려면 어떤 아이템이 필요한가?`라는 질문이 들어오면, GraphRAG는 `용 → 던전 → 필요한 아이템`처럼 연결된 관계를 탐색하고 그 결과를 LLM에 제공합니다.

## 4. 언제 유용한가?

GraphRAG는 다음과 같이 **개체 간 연결과 관계가 중요한 데이터**에 특히 효과적입니다.

- 조직도, 인물 관계, 고객·상품 관계 분석
- 논문, 특허, 뉴스 사이의 연관성 탐색
- 장애 원인 분석, 시스템 의존성 추적
- 여러 문서에 흩어진 정보를 연결해야 하는 질의응답

> GraphRAG가 항상 일반 RAG보다 좋은 것은 아닙니다. 그래프 구축 비용이 들고, 개체·관계 추출이 부정확하면 답변 품질도 낮아질 수 있습니다. 따라서 실제 시스템에서는 **벡터 검색과 그래프 검색을 함께 사용하는 하이브리드 방식**이 자주 활용됩니다.


# 02. GraphRAG 직관 + 옵시디언 스타일 시각화

### 환경

#### 1) `.env`

```
OPENAI_API_KEY=your_openai_api_key
OPENAI_MODEL=gpt-5.5

NEO4J_URI=bolt://localhost:7687
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=graphragpassword
NEO4J_DATABASE=neo4j
```

#### 2) 패키지


In [ ]:
# uv add ipykernel jupyterlab nbformat networkx pyvis langchain langchain-openai langchain-neo4j neo4j pydantic python-dotenv obsidian_graph

## 1. networkx, 노트북 안 그래프

작은 게임 위키를 노드·엣지로 직접 만들어봅니다.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:

nodes = [
    ("마법사", "직업"),
    ("기사", "직업"),
    ("도적", "직업"),
    ("용의 둥지", "던전"),
    ("고블린 산채", "던전"),
    ("화염 드래곤", "보스"),
    ("화염 저항 갑옷", "아이템"),
    ("청룡의 결정", "아이템"),
]



In [ ]:

edges = [
    ("화염 드래곤", "용의 둥지", "등장"),
    ("기사", "용의 둥지", "권장"),
    ("마법사", "용의 둥지", "권장"),
    ("화염 저항 갑옷", "용의 둥지", "필요"),
    ("도적", "고블린 산채", "권장"),
    ("청룡의 결정", "화염 저항 갑옷", "강화"),
]

for s, o, r in edges:
    G.add_edge(s, o, relation=r)

print(f"노드 {G.number_of_nodes()} 개 / 엣지 {G.number_of_edges()} 개")

## 2. 옵시디언 스타일 시각화

`obsidian_graph.show_obsidian_graph` 한 줄로 어두운 배경, 노드 둥둥, 호버 강조 그래프 HTML 생성. 노트북 안 iframe 으로 바로 표시됩니다.

In [ ]:
import sys
sys.path.insert(0, ".")

from obsidian_graph import show_obsidian_graph

show_obsidian_graph(G, output_path="g_full.html", height_px=550)

> 위 iframe 안 노드를 드래그·줌·호버 해보세요. 옵시디언의 그래프 뷰와 동일한 인터랙션.

## 3. 로컬 그래프, 특정 노드 중심 1-hop

옵시디언의 "로컬 그래프" 기능. `highlight_focus=` 인자로 특정 노드 주변만.

In [ ]:
show_obsidian_graph(G, output_path="g_local.html", height_px=400,
                     highlight_focus="용의 둥지")

## 4. 그래프로 관계 추론

질문에서 엔티티 ("용의 둥지") 를 잡아 이웃 노드·엣지를 컨텍스트로 LLM 에 넘김.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = 


def graph_context(graph: nx.DiGraph, entities: list[str]) -> str:
    """엔티티 주변의 노드 속성·관계를 텍스트로."""
    lines = []
    for e in entities:
        if e not in graph.nodes:
            continue
        attrs = ", ".join(f"{k}={v}" for k, v in graph.nodes[e].items())
        lines.append(f"[{e}] {attrs}")
        for nbr in graph.successors(e):
            rel = graph[e][nbr].get("relation", "?")
            lines.append(f"  {e} --{rel}--> {nbr}")
        for nbr in graph.predecessors(e):
            rel = graph[nbr][e].get("relation", "?")
            lines.append(f"  {nbr} --{rel}--> {e}")
    return "\n".join(lines)


PROMPT = ChatPromptTemplate.from_messages([
    ("system", "아래 지식 그래프 정보만 보고 답해. 추측 금지."),
    ("user", "그래프:\n{graph}\n\n질문: {question}"),
])
chain = 

ctx = 

print(chain.invoke({"graph": , "question": " "}))